In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent
COUNTRIES_FILE = REPO_ROOT / "raw_data" / "countries.json"
JSON_FOLDER = REPO_ROOT / "raw_data" / "curricula"
SCHEMA = REPO_ROOT / "database" / "create_schema.sql"
REPORT_FOLDER = REPO_ROOT / "data_load" / "load_reports"

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from cscaf.storage import Storage
from cscaf.study import Study

study = Study.from_config()
storage = Storage.of(study, read_only=False)
if storage.database.create(SCHEMA):
    print("created:", storage.path.relative_to(REPO_ROOT))

In [ ]:
import json

from cscaf.loading import CurriculumLoader

loader = CurriculumLoader(storage)
REPORT_FOLDER.mkdir(parents=True, exist_ok=True)
print("database:", storage.path.relative_to(REPO_ROOT))
print("source  :", JSON_FOLDER.relative_to(REPO_ROOT))

In [ ]:
print("countries inserted:", storage.countries.save_countries(json.loads(COUNTRIES_FILE.read_text(encoding="utf-8"))))
{"country": storage.count_rows("country"), "curriculum": storage.count_rows("curriculum"),
 "syllabus": storage.count_rows("syllabus")}

In [ ]:
documents = CurriculumLoader.read(JSON_FOLDER)
stored = {key for _, key in storage.curricula.curricula() or []}
for document in documents:
    key = document["curriculum_key"]
    print(f"{key:20s} {document['country']['name']:12s} {len(document['courses']):3d} courses"
          + ("   already in the database, will be skipped" if key in stored else ""))
print(len(documents), "curricula")

In [ ]:
report = loader.load(JSON_FOLDER, REPORT_FOLDER)

In [ ]:
problems = report.verify(storage)
print(report.totals())
assert not problems, problems
for c in report.curricula:
    print(f"{c['curriculum_key']:20s} {c['syllabi']:3d} syllabi")
    for line in c["not_english"]:
        print("   ERROR not in English:", line)
assert storage.count_rows("curriculum") == 27 and storage.count_rows("syllabus") == 421